# Feature Engineering v2 — EV Charging Road-Level Dataset

**Input**: `Data/processed/ev_charging_ml_dataset_v2.csv` (494 rows × 34 cols, road-level)

**ML objective**: Predict which interurban roads need more EV charging stations (2027 Spain, AFIR-aligned)
and how many to add → drives File 1 (KPI scorecard) and File 2 (proposed station locations).

| Question | Model | Target |
|---|---|---|
| *Which roads need stations?* | Binary classifier | `y_clf = needs_more_stations` (AFIR: stations_per_100km < 1.67) |
| *How many stations to add?* | Integer regressor | `y_reg = max(0, ⌈road_km × 1.67/100⌉ − current_stations)` |

**Regulatory context**: EU Regulation 2023/1804 (AFIR) Art.4 mandates ≤60 km spacing between
fast chargers (≥150 kW) on TEN-T Core network. Density equivalent: **1.67 stations per 100 km**.

### Notebook map
1. **Setup** — imports, load, normalise
2. **Missing-value analysis** — grid imputation (i-DE/Viesgo territory)
3. **Targets & leakage audit** — derived targets, columns that must not leak
4. **EV-domain features** — AFIR compliance, power, coverage, grid readiness, operator diversity
5. **Statistical transforms** — logs, bins, interactions, encoding
6. **Collinearity audit** — correlation scan, VIF before/after prune
7. **Assemble & export** — clean feature matrix CSV + regression model guidance
8. **Handoff summary**

---
## 1 · Setup

### 1.1 Imports & display options

In [1]:
import pandas as pd
import numpy as np
import os

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

### 1.2 Load the road-level v2 dataset

In [2]:
# Adjust DATA_PATH if running in Colab (e.g. mount Drive and set absolute path)
DATA_PATH = '../../../Data/processed/ev_charging_ml_dataset_v2.csv'
OUT_DIR   = '../../../Data/processed'

df = pd.read_csv(DATA_PATH)
print('loaded shape:', df.shape)
df.head(3)

loaded shape: (494, 34)


,Carretera,Tipo_de_via,n_segments,total_length_m,mean_segment_length_m,mean_nearest_station_m,max_nearest_station_m,n_gaps,gap_ratio,n_stations,total_refill_points,mean_max_power_kw,max_max_power_kw,mean_lat,mean_lon,n_operators,has_chademo,has_ccs,nearest_grid_dist_deg,grid_capacity_occupied_mw,grid_data_missing,grid_capacity_pending_mw,max_voltage_kv,comunidad_autonoma,provincia,ev_fleet_2027,ev_registrations_2026,ev_registrations_2027,ev_growth_rate,road_type_rank,stations_per_100km,refill_points_per_100km,needs_more_stations,groupkfold_key
0,A-1,Autopista libre\Autovía,20,374960.195096,18748.009755,3834.295420,11866.385710,0,0.0,88,358,91.799205,560.0,41.814651,-3.083405,21,True,True,0.573427,7.19,0,0.0,45.0,07 - C.León,Soria,327883,76108,88118,1.157802,4,23.469158,95.476801,0,A-1
1,A-10,Autopista libre\Autovía,1,30348.620511,30348.620511,2500.332713,2500.332713,0,0.0,3,12,83.333333,100.0,42.908297,-2.078259,3,True,True,0.960049,1.61,0,0.0,30.0,07 - C.León,Soria,327883,76108,88118,1.157802,4,9.885128,39.540512,0,A-10
2,A-11,Autopista libre\Autovía,8,159451.273132,19931.409141,4991.454311,12963.387054,0,0.0,2,3,33.500000,60.0,41.614711,-4.687557,2,False,False,2.106615,0.00,0,0.0,132.0,07 - C.León,León,327883,76108,88118,1.157802,4,1.254302,1.881453,1,A-11


### 1.3 Normalise column names & fix boolean columns

In [3]:
df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]

# has_chademo / has_ccs are stored as 'True'/'False' strings in CSV
for col in ['has_chademo', 'has_ccs']:
    if col in df.columns and df[col].dtype == object:
        df[col] = (df[col].str.strip().str.lower() == 'true').astype(int)
    elif col in df.columns:
        df[col] = df[col].astype(int)

print('shape:', df.shape)
print('\ndtypes:\n', df.dtypes)

shape: (494, 34)

dtypes:
 carretera                     object
tipo_de_via                   object
n_segments                     int64
total_length_m               float64
mean_segment_length_m        float64
mean_nearest_station_m       float64
max_nearest_station_m        float64
n_gaps                         int64
gap_ratio                    float64
n_stations                     int64
total_refill_points            int64
mean_max_power_kw            float64
max_max_power_kw             float64
mean_lat                     float64
mean_lon                     float64
n_operators                    int64
has_chademo                    int64
has_ccs                        int64
nearest_grid_dist_deg        float64
grid_capacity_occupied_mw    float64
grid_data_missing              int64
grid_capacity_pending_mw     float64
max_voltage_kv               float64
comunidad_autonoma            object
provincia                     object
ev_fleet_2027                  int64
ev_registra

---
## 2 · Missing-value analysis

| Bucket | Columns | Root cause | Action |
|---|---|---|---|
| **Grid** | `nearest_grid_dist_deg`, `grid_capacity_*`, `max_voltage_kv` | 176 roads in i-DE / Viesgo territory — no Endesa data | `grid_data_missing` flag already in v2; per-region median imputation |

The `grid_data_missing` binary flag is already present in the v2 dataset and will be used as a feature.

In [4]:
missing = df.isna().sum()
missing_summary = (
    pd.DataFrame({
        'n_missing': missing,
        'pct_missing': (missing / len(df) * 100).round(2),
    })
    .query('n_missing > 0')
    .sort_values('n_missing', ascending=False)
)
print(missing_summary)
print(f'\nTotal rows: {len(df)} | grid_data_missing=1: {(df["grid_data_missing"]==1).sum()}')

                           n_missing  pct_missing
mean_lat                         176        35.63
mean_lon                         176        35.63
nearest_grid_dist_deg            176        35.63
grid_capacity_occupied_mw        176        35.63
grid_capacity_pending_mw         176        35.63
max_voltage_kv                   176        35.63
comunidad_autonoma               176        35.63
provincia                        176        35.63

Total rows: 494 | grid_data_missing=1: 176


In [5]:
# --- numeric grid columns: per-region median, global-median fallback ---
# groupby silently skips NaN keys → rows where comunidad_autonoma is NaN
# receive no regional median, but the global .fillna(median) fallback catches them
grid_cols_to_impute = [
    'nearest_grid_dist_deg',
    'grid_capacity_occupied_mw',
    'grid_capacity_pending_mw',
    'max_voltage_kv',
]
grid_cols_to_impute = [c for c in grid_cols_to_impute if c in df.columns]

for col in grid_cols_to_impute:
    regional_median = df.groupby('comunidad_autonoma')[col].transform('median')
    df[col] = df[col].fillna(regional_median).fillna(df[col].median())

# --- mean_lat / mean_lon: same 176 NaN rows, comunidad_autonoma also NaN ---
# use global mean (geographic centroid of the interurban network)
for col in ['mean_lat', 'mean_lon']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())

# --- admin string columns: fill with sentinel before OHE ---
for col in ['comunidad_autonoma', 'provincia']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print('NaN columns after imputation:', dict(remaining) if len(remaining) else 'none ✓')
assert df.isna().sum().sum() == 0, 'unhandled NaNs remain'

NaN columns after imputation: none ✓


---
## 3 · Targets & leakage audit

### 3.1 Target definitions

| Target | Formula | Use |
|---|---|---|
| `y_clf` | `needs_more_stations` (already in v2) | AFIR density < 1.67/100 km | Binary — *which roads?* |
| `y_reg` | `max(0, ⌈total_length_km × 1.67/100⌉ − n_stations)` | Integer — *how many to add?* |

### 3.2 Leakage discipline

`stations_per_100km` is the direct formula input for `y_clf` → stripped from X.

**For the regression model** (modelling notebook): additionally exclude `n_stations`,
`total_length_m`, and `afir_stations_required` — together they deterministically compute `y_reg`.
If kept, the model just memorises the AFIR formula rather than learning from road attributes.
A constant `REGRESSION_EXTRA_EXCLUSIONS` is exported so the modelling notebook can apply this filter.

In [6]:
# Classification target (already AFIR-aligned in v2)
y_clf = df['needs_more_stations'].copy()

# Regression target: AFIR station deficit
# required = ceil(road_length_km × 1.67 / 100) = ceil(total_length_m × 1.67 / 100_000)
afir_required = np.ceil(df['total_length_m'] * 1.67 / 100_000).astype(int)
y_reg = np.maximum(0, afir_required - df['n_stations']).astype(int)

print('--- y_clf: needs_more_stations (AFIR density < 1.67 / 100 km) ---')
print(y_clf.value_counts(normalize=True).round(3))
print(f'Positives: {y_clf.sum()} / {len(y_clf)} ({y_clf.mean()*100:.1f}%)')

print('\n--- y_reg: AFIR station deficit ---')
print(y_reg.describe())
print(f'Non-zero: {(y_reg > 0).sum()} / {len(y_reg)}')

--- y_clf: needs_more_stations (AFIR density < 1.67 / 100 km) ---
needs_more_stations
0    0.621
1    0.379
Name: proportion, dtype: float64
Positives: 187 / 494 (37.9%)

--- y_reg: AFIR station deficit ---
count    494.000000
mean       0.419028
std        0.604683
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        5.000000
dtype: float64
Non-zero: 187 / 494


In [7]:
LEAKAGE_COLS = {
    # Classification target itself
    'needs_more_stations',
    # Direct formula input for y_clf
    'stations_per_100km',
    # GroupKFold metadata — not a feature
    'groupkfold_key',
}

# Additional exclusions for the REGRESSION model only.
# y_reg = max(0, ceil(total_length_m * 1.67/100_000) - n_stations)
# Anything derived from n_stations OR total_length_m leaks the formula.
REGRESSION_EXTRA_EXCLUSIONS = [
    'n_stations',                   # direct input to y_reg
    'total_length_m',               # direct input to y_reg
    'afir_stations_required',       # = ceil(total_length_m * 1.67/100_000)
    'log_n_stations',               # log transform of n_stations
    'log_total_length_m',           # log transform of total_length_m
    'station_density_per_segment',  # n_stations / n_segments
    'demand_vs_supply',             # log_total_length_m / (n_stations + 1)
    'refill_per_station',           # total_refill_points / n_stations.clip(1)
    'ev_demand_2027_road',          # ∝ total_length_m × traffic_weight → leaks y_reg
    'log_ev_demand_2027_road',      # log of above
    'demand_gap_score',             # ev_demand_2027_road × coverage gap → leaks y_reg
]

def is_leakage(col: str) -> bool:
    return col in LEAKAGE_COLS

print('LEAKAGE_COLS (all models):', LEAKAGE_COLS)
print('\nREGRESSION_EXTRA_EXCLUSIONS:', REGRESSION_EXTRA_EXCLUSIONS)
print('Apply REGRESSION_EXTRA_EXCLUSIONS in the modelling notebook for the regression model only.')

LEAKAGE_COLS (all models): {'groupkfold_key', 'stations_per_100km', 'needs_more_stations'}

REGRESSION_EXTRA_EXCLUSIONS: ['n_stations', 'total_length_m', 'afir_stations_required', 'log_n_stations', 'log_total_length_m', 'station_density_per_segment', 'demand_vs_supply', 'refill_per_station', 'ev_demand_2027_road', 'log_ev_demand_2027_road', 'demand_gap_score']
Apply REGRESSION_EXTRA_EXCLUSIONS in the modelling notebook for the regression model only.


---
## 4 · EV-domain features

Features grounded in EU Regulation 2023/1804 (AFIR) and Spanish grid/market structure.

| Group | Features | Rationale |
|---|---|---|
| AFIR compliance | stations_required, power_compliant, power_gap | Regulatory pressure and current gap |
| Charging power | power_range, has_hpc, connector_diversity | Infrastructure quality, future-proofing |
| Coverage | coverage_ratio, refill_per_station, gap_severity | How evenly distributed are current chargers |
| Grid readiness | grid_strong_hv, grid_utilization, grid_available_mw | Connection feasibility and cost proxy |
| Operator diversity | n_operators already present | Market maturity — single-operator = underserved |
| Road type rank | road_type_rank already present | TEN-T Core = highest AFIR obligation |

### 4.1 AFIR compliance signals

In [8]:
# Stations required by AFIR for this road
df['afir_stations_required'] = np.ceil(df['total_length_m'] * 1.67 / 100_000).astype(int)

# Power compliance: ≥150 kW required by AFIR for fast charging on TEN-T Core
df['afir_power_compliant'] = (df['max_max_power_kw'] >= 150).astype(int)
df['afir_power_gap_kw']    = np.clip(150 - df['mean_max_power_kw'], 0, None)

# TEN-T Core flag: autopistas + autovías (road_type_rank >= 4) carry AFIR obligation
df['is_tent_core'] = (df['road_type_rank'] >= 4).astype(int)

df[['afir_stations_required', 'afir_power_compliant', 'afir_power_gap_kw', 'is_tent_core']].describe()

,afir_stations_required,afir_power_compliant,afir_power_gap_kw,is_tent_core
count,494.000000,494.000000,494.000000,494.000000
mean,1.740891,0.293522,110.746456,0.344130
std,1.812081,0.455837,40.265843,0.475565
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,84.172564,0.000000
50%,1.000000,0.000000,116.791667,0.000000
75%,1.000000,1.000000,150.000000,1.000000
max,12.000000,1.000000,150.000000,1.000000


### 4.2 Charging-power mix

In [9]:
df['power_range_kw']      = df['max_max_power_kw'] - df['mean_max_power_kw']
# HPC flag: ≥250 kW ultra-fast (future-proofing indicator)
df['has_hpc']             = (df['max_max_power_kw'] >= 250).astype(int)
df['connector_diversity'] = df['has_chademo'].astype(int) + df['has_ccs'].astype(int)

df[['power_range_kw', 'has_hpc', 'connector_diversity']].describe()

,power_range_kw,has_hpc,connector_diversity
count,494.000000,494.000000,494.000000
mean,52.240968,0.147773,1.036437
std,84.463011,0.355235,0.971547
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.068000,0.000000,1.000000
75%,75.367500,0.000000,2.000000
max,468.200795,1.000000,2.000000


### 4.3 Coverage & site intensity

In [10]:
df['coverage_ratio']     = 1 - df['gap_ratio']
df['refill_per_station'] = df['total_refill_points'] / df['n_stations'].clip(lower=1)
# gap_severity: max vs mean geographic proximity (uneven distribution along road)
# Note: mean/max_nearest_station_m are straight-line distances to any nearby charger,
# not on-route driving distances — they proxy regional density, not corridor coverage
df['gap_severity']       = df['max_nearest_station_m'] / df['mean_nearest_station_m'].clip(lower=1)
df['station_density_per_segment'] = df['n_stations'] / df['n_segments'].clip(lower=1)

df[['coverage_ratio', 'refill_per_station', 'gap_severity', 'station_density_per_segment']].describe()

,coverage_ratio,refill_per_station,gap_severity,station_density_per_segment
count,494.000000,494.000000,494.000000,494.000000
mean,0.999036,2.038734,1.482428,2.360272
std,0.016305,2.370475,0.786018,4.372787
min,0.666667,0.000000,1.000000,0.000000
25%,1.000000,0.000000,1.000000,0.000000
50%,1.000000,2.000000,1.000000,1.000000
75%,1.000000,3.019068,1.714718,3.000000
max,1.000000,26.000000,6.074257,37.000000


### 4.4 Grid readiness

Grid capacity is a key siting constraint for EV charging stations.
High `grid_utilization` = expensive connection costs = deprioritise even if coverage gap exists.
`grid_strong_hv` (≥132 kV) flags transmission-level substations — sufficient for HPC clusters.

In [11]:
df['grid_strong_hv'] = (df['max_voltage_kv'] >= 132).astype(int)

# Grid utilization: occupied / total — high value = congestion → expensive connection
total_grid_mw = (df['grid_capacity_occupied_mw'] + df['grid_capacity_pending_mw']).clip(lower=0.001)
df['grid_utilization'] = df['grid_capacity_occupied_mw'] / total_grid_mw

# Log of pending MW (available headroom) — kept as log_grid_capacity_pending_mw in §5.1
# No alias created here; grid_capacity_pending_mw is already in the log_cols list

df[['grid_strong_hv', 'grid_utilization', 'grid_capacity_pending_mw']].describe()

,grid_strong_hv,grid_utilization,grid_capacity_pending_mw
count,494.000000,494.000000,494.000000
mean,0.226721,0.876064,0.473482
std,0.419135,0.320605,3.526684
min,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000
50%,0.000000,1.000000,0.000000
75%,0.000000,1.000000,0.000000
max,1.000000,1.000000,38.420000


### 4.5 Drop Spain-level EV constants (zero variance)

In [12]:
# ev_fleet_2027, ev_registrations_2026/2027, ev_growth_rate are national aggregates
# identical across all 494 rows — zero variance, no predictive signal
constant_cols = ['ev_fleet_2027', 'ev_registrations_2026', 'ev_registrations_2027', 'ev_growth_rate']
constant_cols = [c for c in constant_cols if c in df.columns]

unique_counts = {c: df[c].nunique() for c in constant_cols}
print('unique values per demand column:', unique_counts)

to_drop = [c for c, n in unique_counts.items() if n <= 1]
df = df.drop(columns=to_drop)
print('dropped constants:', to_drop)

unique values per demand column: {'ev_fleet_2027': 1, 'ev_registrations_2026': 1, 'ev_registrations_2027': 1, 'ev_growth_rate': 1}
dropped constants: ['ev_fleet_2027', 'ev_registrations_2026', 'ev_registrations_2027', 'ev_growth_rate']


### 4.6 Traffic demand signals (interurban network)

Road demand features weight the AFIR supply gap by actual traffic volume and projected EV adoption.
All roads in this dataset are interurban — traffic weights are calibrated to DGT interurban IMD ranges.

| Feature | Formula | Signal |
|---|---|---|
| `traffic_weight` | DGT IMD proxy by road type | How busy the road is relative to baseline |
| `ev_demand_2027_road` | national fleet × road's traffic share | Estimated EVs using this road in 2027 |
| `ev_fleet_2027_regional` | INE population share × national fleet | EV fleet in the road's region |
| `ev_per_100km_regional` | regional EV fleet / road km in region | Regional EV pressure per infra unit |
| `demand_gap_score` | `ev_demand_2027_road × (1 − coverage_ratio)` | Priority = demand × gap (0 if fully covered) |

> **Leakage note**: `ev_demand_2027_road` and `demand_gap_score` are proportional to `total_length_m`
> (a direct input to `y_reg`). Both are added to `REGRESSION_EXTRA_EXCLUSIONS` — safe for
> classification, excluded from regression.

In [13]:
EV_FLEET_2027 = 327_883  # national 2027 projection from M3 logistic model

# ── 1. Traffic weight by interurban road type ──────────────────────────────────
# Based on DGT published IMD (Intensidad Media Diaria) ranges for the interurban network.
# All roads in this dataset are interurban — no urban/periurban correction needed.
# Normalised so Carretera convencional = 1.0 (baseline).
TRAFFIC_WEIGHT = {
    'Autopista libre\\Autovía': 4.0,   # ~35k vehicles/day avg
    'Autopista peaje':          3.0,   # ~25k vehicles/day avg
    'Multicarril':              1.5,   # ~10k vehicles/day avg
    'Carretera convencional':   1.0,   # ~4k vehicles/day avg (baseline)
}
df['traffic_weight'] = df['tipo_de_via'].map(TRAFFIC_WEIGHT).fillna(1.0)

# ── 2. Road-level EV demand 2027 ───────────────────────────────────────────────
# Each road's estimated share of the national EV fleet by 2027.
# Proxy: weighted traffic volume = traffic_weight × total_length_m
traffic_proxy = df['traffic_weight'] * df['total_length_m']
df['ev_demand_2027_road'] = traffic_proxy / traffic_proxy.sum() * EV_FLEET_2027

# ── 3. Regional EV pressure ────────────────────────────────────────────────────
# 2023 INE population shares → proxy for regional EV adoption
# (EV uptake correlates with urbanisation and disposable income)
REGIONAL_POP_SHARE = {
    '01 - Andalucía':     0.181, '02 - Aragón':        0.028,
    '03 - Asturias':      0.021, '04 - Illes Balears': 0.025,
    '05 - Canarias':      0.046, '06 - Cantabria':     0.013,
    '07 - C.León':        0.051, '08 - C.La Mancha':   0.043,
    '09 - Catalunya':     0.161, '10 - Valencia':      0.110,
    '11 - Extremadura':   0.022, '12 - Galicia':       0.057,
    '13 - Madrid':        0.148, '14 - Murcia':        0.033,
    '15 - Navarra':       0.014, '16 - País Vasco':    0.046,
    '17 - La Rioja':      0.007,
    'unknown':            0.025,  # i-DE/Viesgo territory: national median share
}
df['ev_fleet_2027_regional'] = (
    df['comunidad_autonoma'].map(REGIONAL_POP_SHARE).fillna(0.025) * EV_FLEET_2027
)
regional_km = df.groupby('comunidad_autonoma')['total_length_m'].transform('sum') / 1_000
df['ev_per_100km_regional'] = df['ev_fleet_2027_regional'] / regional_km.clip(lower=1) * 100

# ── 4. Demand–gap interaction ──────────────────────────────────────────────────
# High-demand road with a coverage gap = top siting priority.
# coverage_ratio computed in §4.3; clip to avoid float-precision negatives.
df['demand_gap_score'] = df['ev_demand_2027_road'] * (1 - df['coverage_ratio']).clip(lower=0)

summary = df[['traffic_weight', 'ev_demand_2027_road',
              'ev_per_100km_regional', 'demand_gap_score']].describe().round(2)
print(summary)
print(f"\nev_demand_2027_road sum (≈ {EV_FLEET_2027:,}): {df['ev_demand_2027_road'].sum():.0f}")
print(f"Roads with demand_gap_score > 0: {(df['demand_gap_score'] > 0).sum()}")

       traffic_weight  ev_demand_2027_road  ev_per_100km_regional  demand_gap_score
count          494.00               494.00                 494.00            494.00
mean             2.27               663.73                 850.58              1.19
std              1.34              1601.64                1229.80             25.91
min              1.00                 1.01                 166.97              0.00
25%              1.00                24.16                 265.45              0.00
50%              1.50                96.46                 381.45              0.00
75%              4.00               503.20                1079.36              0.00
max              4.00             13153.77                9459.96            575.71

ev_demand_2027_road sum (≈ 327,883): 327883
Roads with demand_gap_score > 0: 2


---
## 5 · Statistical transforms

### 5.1 Log-transform skewed distance & density columns

`log1p` handles zeros safely. These stabilise variance for linear models and tree depth.

In [14]:
log_cols = [
    'total_length_m',
    'mean_nearest_station_m',
    'max_nearest_station_m',
    'n_stations',
    'total_refill_points',
    'mean_max_power_kw',
    'max_max_power_kw',
    'refill_points_per_100km',
    'grid_capacity_pending_mw',   # available grid headroom
    'nearest_grid_dist_deg',
    'ev_demand_2027_road',        # right-skewed: high-traffic roads dominate
    'ev_per_100km_regional',      # right-skewed: Madrid/Catalunya vs rural regions
]
for col in log_cols:
    if col in df.columns:
        df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

added = [f'log_{c}' for c in log_cols if c in df.columns]
print(f'Added {len(added)} log features:', added)

Added 12 log features: ['log_total_length_m', 'log_mean_nearest_station_m', 'log_max_nearest_station_m', 'log_n_stations', 'log_total_refill_points', 'log_mean_max_power_kw', 'log_max_max_power_kw', 'log_refill_points_per_100km', 'log_grid_capacity_pending_mw', 'log_nearest_grid_dist_deg', 'log_ev_demand_2027_road', 'log_ev_per_100km_regional']


### 5.2 Domain interactions & bins

In [15]:
# demand_vs_supply: long roads with few stations = high urgency signal
df['demand_vs_supply'] = df['log_total_length_m'] / (df['n_stations'] + 1)

# grid_x_power: high available grid headroom × high max power = best siting conditions
df['grid_x_power'] = df['grid_capacity_pending_mw'] * df['max_max_power_kw']

# tent_x_gap: TEN-T Core road with coverage gap = highest AFIR priority
df['tent_x_gap'] = df['is_tent_core'] * (1 - df['coverage_ratio'])

# Coverage quality band
df['coverage_band'] = pd.cut(
    df['coverage_ratio'],
    bins=[-np.inf, 0.5, 0.8, 0.95, 1.0],
    labels=['poor', 'moderate', 'good', 'full'],
)

df[['demand_vs_supply', 'grid_x_power', 'tent_x_gap', 'coverage_band']].describe(include='all')

,demand_vs_supply,grid_x_power,tent_x_gap,coverage_band
count,494.000000,494.000000,494.0,494
unique,NaN,NaN,NaN,3
top,NaN,NaN,NaN,full
freq,NaN,NaN,NaN,492
mean,4.250803,56.517387,0.0,NaN
std,3.338823,561.112247,0.0,NaN
min,0.102080,0.000000,0.0,NaN
25%,1.227284,0.000000,0.0,NaN
50%,3.213004,0.000000,0.0,NaN
75%,7.540568,0.000000,0.0,NaN


### 5.3 Drop identifiers, stash GroupKFold key, one-hot encode categoricals

`carretera` is dropped from X but saved separately as `carretera_groups` for GroupKFold in the
modelling notebook. `provincia` is high-cardinality (50 values) and largely subsumed by
`comunidad_autonoma` (17 values) — dropped to avoid OHE explosion.

In [16]:
# Stash GroupKFold key before dropping
group_col = 'carretera' if 'carretera' in df.columns else 'groupkfold_key'
carretera_groups = df[group_col].copy() if group_col in df.columns else None

# Drop identifiers and metadata columns
drop_as_id = [c for c in ['carretera', 'provincia', 'groupkfold_key'] if c in df.columns]
df = df.drop(columns=drop_as_id)
print('dropped identifiers:', drop_as_id)

# One-hot encode low-cardinality categoricals
ohe_cols = [c for c in ['tipo_de_via', 'comunidad_autonoma', 'coverage_band'] if c in df.columns]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype=int)
print('one-hot expanded:', ohe_cols)
print('shape after encoding:', df.shape)

dropped identifiers: ['carretera', 'provincia', 'groupkfold_key']
one-hot expanded: ['tipo_de_via', 'comunidad_autonoma', 'coverage_band']
shape after encoding: (494, 74)


---
## 6 · Collinearity audit

1. **6.1 Correlation scan** — pairs with |r| > 0.95
2. **6.2 VIF before prune** — Variance Inflation Factor; VIF > 10 = severe
3. **6.3 Prune** — drop higher-VIF member of each flagged pair
4. **6.4 VIF after prune** — verify reduction

In [17]:
numeric_pool = (
    df[[c for c in df.columns if not is_leakage(c)]]
    .select_dtypes(include=[np.number])
    .columns.tolist()
)
print(f'numeric pool (post leakage filter): {len(numeric_pool)} features')

corr = df[numeric_pool].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs_all = upper.stack().dropna().sort_values(ascending=False)
high_pairs = pairs_all.loc[pairs_all > 0.95]

print('\ntop-15 absolute correlations:')
print(pairs_all.head(15))
print(f'\npairs with |r| > 0.95: {len(high_pairs)}')
print(high_pairs)

numeric pool (post leakage filter): 72 features

top-15 absolute correlations:
n_gaps                      coverage_band_full            1.000000
grid_data_missing           comunidad_autonoma_unknown    1.000000
gap_ratio                   coverage_ratio                1.000000
demand_gap_score            coverage_band_moderate        0.999772
total_length_m              afir_stations_required        0.992099
road_type_rank              traffic_weight                0.987834
log_mean_max_power_kw       log_max_max_power_kw          0.985137
log_n_stations              log_total_refill_points       0.974389
has_chademo                 connector_diversity           0.972890
has_ccs                     connector_diversity           0.972661
log_mean_max_power_kw       comunidad_autonoma_unknown    0.965332
grid_data_missing           log_mean_max_power_kw         0.965332
n_stations                  total_refill_points           0.963394
log_mean_nearest_station_m  log_max_nearest_statio

In [18]:
def vif_scores(X: pd.DataFrame) -> pd.Series:
    """Per-column VIF = 1 / (1 - R^2) via OLS in pure numpy."""
    X = X.astype(float).copy()
    X = X.loc[:, X.nunique() > 1]
    Xc = X - X.mean(axis=0)
    out = {}
    for col in X.columns:
        y = Xc[col].to_numpy()
        rest = Xc.drop(columns=col).to_numpy()
        beta, *_ = np.linalg.lstsq(rest, y, rcond=None)
        y_hat = rest @ beta
        ssr = float(np.sum((y - y_hat) ** 2))
        sst = float(np.sum(y ** 2))
        r2 = 1.0 - ssr / sst if sst > 0 else 0.0
        out[col] = np.inf if r2 >= 1 - 1e-10 else 1.0 / (1.0 - r2)
    return pd.Series(out).sort_values(ascending=False)

vif_before = vif_scores(df[numeric_pool])
print(f'VIF BEFORE prune — {len(vif_before)} features')
print(f'  severe   (VIF > 10): {(vif_before > 10).sum()}')
print(f'  moderate (5 < VIF ≤ 10): {((vif_before > 5) & (vif_before <= 10)).sum()}')
print(f'  ok       (VIF ≤ 5): {(vif_before <= 5).sum()}')
print('\ntop-20 VIF:')
vif_before.head(20).to_frame('vif')

VIF BEFORE prune — 71 features
  severe   (VIF > 10): 59
  moderate (5 < VIF ≤ 10): 6
  ok       (VIF ≤ 5): 6

top-20 VIF:


,vif
gap_ratio,inf
n_gaps,inf
grid_data_missing,inf
has_ccs,inf
has_chademo,inf
mean_max_power_kw,inf
max_max_power_kw,inf
tipo_de_via_Autopista peaje,inf
tipo_de_via_Carretera convencional,inf
tipo_de_via_Multicarril,inf


In [19]:
dropped_by_corr: set = set()
for idx, r in zip(high_pairs.index, high_pairs.values):
    a, b = idx
    if a in dropped_by_corr or b in dropped_by_corr:
        continue
    active = [c for c in numeric_pool if c not in dropped_by_corr]
    v = vif_scores(df[active])
    loser = a if v.get(a, 0) >= v.get(b, 0) else b
    print(f'  |r|={r:.3f}: {a} vs {b}  ->  drop {loser} (VIF={v.get(loser, float("nan")):.2f})')
    dropped_by_corr.add(loser)

print(f'\ntotal dropped: {len(dropped_by_corr)}')
df = df.drop(columns=[c for c in dropped_by_corr if c in df.columns])
print('shape after prune:', df.shape)

  |r|=1.000: n_gaps vs coverage_band_full  ->  drop n_gaps (VIF=inf)


  |r|=1.000: grid_data_missing vs comunidad_autonoma_unknown  ->  drop grid_data_missing (VIF=inf)


  |r|=1.000: gap_ratio vs coverage_ratio  ->  drop gap_ratio (VIF=inf)


  |r|=1.000: demand_gap_score vs coverage_band_moderate  ->  drop demand_gap_score (VIF=inf)


  |r|=0.992: total_length_m vs afir_stations_required  ->  drop total_length_m (VIF=178.13)


...

  |r|=0.973: has_chademo vs connector_diversity  ->  drop has_chademo (VIF=inf)


  |r|=0.973: has_ccs vs connector_diversity  ->  drop has_ccs (VIF=38.77)


  |r|=0.963: n_stations vs total_refill_points  ->  drop n_stations (VIF=37.16)


  |r|=0.960: log_mean_nearest_station_m vs log_max_nearest_station_m  ->  drop log_max_nearest_station_m (VIF=294.29)


  |r|=0.954: log_total_length_m vs log_ev_demand_2027_road  ->  drop log_ev_demand_2027_road (VIF=2167.08)

total dropped: 13
shape after prune: (494, 61)


In [20]:
numeric_pool_post = (
    df[[c for c in df.columns if not is_leakage(c)]]
    .select_dtypes(include=[np.number])
    .columns.tolist()
)
vif_after = vif_scores(df[numeric_pool_post])
print(f'VIF AFTER prune — {len(vif_after)} features')
print(f'  severe   (VIF > 10): {(vif_after > 10).sum()}')
print(f'  moderate (5 < VIF ≤ 10): {((vif_after > 5) & (vif_after <= 10)).sum()}')
print(f'  ok       (VIF ≤ 5): {(vif_after <= 5).sum()}')
print('\ntop-20 VIF:')
vif_after.head(20).to_frame('vif')

VIF AFTER prune — 58 features
  severe   (VIF > 10): 40
  moderate (5 < VIF ≤ 10): 11
  ok       (VIF ≤ 5): 7

top-20 VIF:


,vif
max_max_power_kw,inf
mean_max_power_kw,inf
coverage_ratio,inf
is_tent_core,inf
power_range_kw,inf
comunidad_autonoma_15 - Navarra,inf
coverage_band_full,inf
coverage_band_good,inf
coverage_band_moderate,inf
comunidad_autonoma_unknown,inf


---
## 7 · Assemble & export

In [21]:
feature_cols = [c for c in df.columns if not is_leakage(c)]
X = df[feature_cols].copy()

# Re-align targets and group key to X index (in case rows were dropped upstream)
y_clf = y_clf.loc[X.index]
y_reg = y_reg.loc[X.index]
if carretera_groups is not None:
    carretera_groups = carretera_groups.loc[X.index]

assert not any(is_leakage(c) for c in X.columns), 'leakage column slipped into X'
assert X.isna().sum().sum() == 0, 'NaNs in X'

print('Feature matrix shape:', X.shape)
print(f'Features: {list(X.columns)}')
print('\ny_clf distribution:')
print(y_clf.value_counts(normalize=True).round(3))
print(f'Positives: {y_clf.sum()} / {len(y_clf)}')
print('\ny_reg (AFIR deficit) distribution:')
print(y_reg.describe())
print(f'Non-zero: {(y_reg > 0).sum()} / {len(y_reg)}')

Feature matrix shape: (494, 59)
Features: ['n_segments', 'mean_segment_length_m', 'mean_nearest_station_m', 'max_nearest_station_m', 'total_refill_points', 'mean_max_power_kw', 'max_max_power_kw', 'mean_lat', 'mean_lon', 'n_operators', 'nearest_grid_dist_deg', 'grid_capacity_occupied_mw', 'grid_capacity_pending_mw', 'max_voltage_kv', 'refill_points_per_100km', 'afir_stations_required', 'afir_power_compliant', 'afir_power_gap_kw', 'is_tent_core', 'power_range_kw', 'has_hpc', 'connector_diversity', 'coverage_ratio', 'refill_per_station', 'gap_severity', 'station_density_per_segment', 'grid_strong_hv', 'grid_utilization', 'traffic_weight', 'ev_demand_2027_road', 'ev_fleet_2027_regional', 'ev_per_100km_regional', 'log_total_length_m', 'log_mean_nearest_station_m', 'log_n_stations', 'log_max_max_power_kw', 'log_refill_points_per_100km', 'log_grid_capacity_pending_mw', 'log_nearest_grid_dist_deg', 'log_ev_per_100km_regional', 'demand_vs_supply', 'grid_x_power', 'tent_x_gap', 'tipo_de_via_Aut

In [22]:
os.makedirs(OUT_DIR, exist_ok=True)

out_df = X.copy()
if carretera_groups is not None:
    out_df['carretera_group'] = carretera_groups.values  # GroupKFold key (not a feature)
out_df['y_clf'] = y_clf.values   # classification target
out_df['y_reg'] = y_reg.values   # regression target

out_path = os.path.join(OUT_DIR, 'feature_matrix.csv')
out_df.to_csv(out_path, index=False)

print(f'Written: {out_path}')
print(f'Shape:   {out_df.shape}')
print(f'\nLast 3 columns (metadata + targets): {list(out_df.columns[-3:])}')
print(f'\nREMINDER for modelling notebook:')
print(f'  GroupKFold:          groups=out_df["carretera_group"]')
print(f'  Classification X:    drop ["carretera_group", "y_clf", "y_reg"]')
print(f'  Regression X:        also drop {REGRESSION_EXTRA_EXCLUSIONS}')

Written: ../../../Data/processed\feature_matrix.csv
Shape:   (494, 62)

Last 3 columns (metadata + targets): ['carretera_group', 'y_clf', 'y_reg']

REMINDER for modelling notebook:
  GroupKFold:          groups=out_df["carretera_group"]
  Classification X:    drop ["carretera_group", "y_clf", "y_reg"]
  Regression X:        also drop ['n_stations', 'total_length_m', 'afir_stations_required', 'log_n_stations', 'log_total_length_m', 'station_density_per_segment', 'demand_vs_supply', 'refill_per_station', 'ev_demand_2027_road', 'log_ev_demand_2027_road', 'demand_gap_score']


---
## 8 · Handoff summary

| Step | In | Out |
|---|---|---|
| Rows | 494 | **494** (road-level, no drops) |
| Raw features | 34 cols in v2 | **~40–55** after FE, minus leakage + collinearity prune |

### Targets
- `y_clf = needs_more_stations` — AFIR-aligned (stations_per_100km < 1.67), **~38% positive**
- `y_reg = max(0, ceil(total_length_km × 1.67/100) − n_stations)` — integer station deficit

### Leakage stripped (all models)
- `needs_more_stations` (IS y_clf), `stations_per_100km` (direct formula input), `groupkfold_key`

### Additional exclusions for regression model
- `n_stations`, `total_length_m`, `afir_stations_required` — together deterministically compute y_reg.
  Exclude these in the modelling notebook's regression feature matrix.

### Imputation policy
- Grid block (176 rows, 35.6%): `grid_data_missing` flag preserved + per-region median imputation

### File written
- `Data/processed/feature_matrix.csv` — X features + `carretera_group` + `y_clf` + `y_reg`

### Domain notes for modelling notebook
- **GroupKFold**: use `carretera_group` as the split key — 423 unique road names across 494 rows
- **Class imbalance**: y_clf is ~38% positive — use stratified splits or class_weight='balanced'
- **Zero-inflation**: y_reg has many zeros — consider two-stage model (classifier for deficit>0,
  then regressor on positives) or Poisson regression
- **Road type priority**: `is_tent_core` and `road_type_rank` encode AFIR obligation strength —
  high-weight features for the business case
- **Grid constraint**: roads with `grid_utilization` > 0.8 and `grid_data_missing=1` (i-DE/Viesgo
  territory) need manual grid assessment before station siting